<a href="https://colab.research.google.com/github/FatimaMirandap/Demo/blob/main/homework3_parte1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 3 — Elastic-Net for Regression
**CentroGeo · Machine Learning · 2026**

## Parte I: Datos sintéticos con colinealidad controlada

**Objetivo:** construir un dataset sintético con grupos de variables altamente correlacionadas, donde solo unas pocas explican la respuesta, y comparar OLS, Ridge, LASSO y Elastic-Net contra la verdad conocida.

---
> **Nota sobre notación sklearn vs PDF**  
> En `ElasticNet(alpha=a, l1_ratio=r)` de sklearn:
> - `alpha` = λ (fuerza de regularización)  
> - `l1_ratio` = α (mezcla L1/L2)  
> α=1 → LASSO puro · α=0 → Ridge puro

## 0. Instalación e importaciones

In [ ]:
# Instalar el paquete geoml del profesor (solo necesario en Colab)
!pip install -q git+https://github.com/hucarlos08/GEO-ML.git

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import (
    LinearRegression, Ridge, RidgeCV,
    Lasso, LassoCV,
    ElasticNet, ElasticNetCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid', palette='muted')
print('Librerias cargadas correctamente v')

---
## 1. Generación de datos sintéticos

Diseño según el PDF:
- `n=1000` observaciones, `p=50` predictores
- 3 variables latentes z1, z2, z3 ~ N(0,1)
- **Grupo A** (relevante, colineal): x1, x2, x3 ≈ z1  
- **Grupo B** (relevante, colineal): x4, x5 ≈ z2  
- **Grupo C** (colineal pero irrelevante): x6, x7, x8 ≈ z3  
- **x9**: señal aislada, sin colinealidad  
- **x10…x50**: ruido puro  
- Respuesta: `y = 3*x1 - 2*x4 + 1.5*x9 + eta`,  eta ~ N(0,1)

In [ ]:
def generate_collinear_data(n=1000, p=50, seed=42):
    """
    Genera datos sinteticos con colinealidad controlada segun el Homework 3.

    Grupos:
        A (x1-x3):  relevante, rho ~= 0.99
        B (x4-x5):  relevante, rho ~= 0.99
        C (x6-x8):  irrelevante, rho ~= 0.99
        x9:         senal aislada
        x10-x50:    ruido puro

    Verdad: y = 3*x1 - 2*x4 + 1.5*x9 + N(0,1)
    """
    rng = np.random.default_rng(seed)

    # Variables latentes
    z1, z2, z3 = rng.standard_normal((3, n))

    # Ruido intra-grupo: epsilon ~ N(0, 0.1^2) -> correlacion ~= 0.99
    eps = rng.normal(0, 0.1, (8, n))

    # Grupos correlacionados
    x1 = z1 + eps[0]   # Grupo A
    x2 = z1 + eps[1]
    x3 = z1 + eps[2]
    x4 = z2 + eps[3]   # Grupo B
    x5 = z2 + eps[4]
    x6 = z3 + eps[5]   # Grupo C (irrelevante)
    x7 = z3 + eps[6]
    x8 = z3 + eps[7]

    # Senal aislada y ruido
    x9 = rng.standard_normal(n)
    noise_vars = rng.standard_normal((40, n))   # x10 ... x50

    # Matriz de diseno X (n x 50)
    X = np.column_stack([x1, x2, x3, x4, x5, x6, x7, x8, x9, noise_vars.T])

    # Respuesta
    eta = rng.standard_normal(n)
    y   = 3*x1 - 2*x4 + 1.5*x9 + eta

    # Coeficientes verdaderos (para evaluacion)
    beta_true = np.zeros(p)
    beta_true[0] = 3.0    # x1
    beta_true[3] = -2.0   # x4
    beta_true[8] = 1.5    # x9

    feature_names = (
        ['x1_A', 'x2_A', 'x3_A',
         'x4_B', 'x5_B',
         'x6_C', 'x7_C', 'x8_C',
         'x9_iso'] +
        [f'x{i}_noise' for i in range(10, 51)]
    )

    return X, y, beta_true, feature_names


X, y, beta_true, feature_names = generate_collinear_data()
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print('Coeficientes verdaderos no nulos:',
      [(feature_names[i], v) for i, v in enumerate(beta_true) if v != 0])

In [ ]:
# Verificar correlaciones intra-grupo
df_check = pd.DataFrame(X[:, :9], columns=feature_names[:9])
corr = df_check.corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Correlacion entre las primeras 9 variables', fontsize=12)
plt.tight_layout()
plt.show()

print(f"Corr. media Grupo A: {corr.loc[['x1_A','x2_A','x3_A'],['x1_A','x2_A','x3_A']].values[np.triu_indices(3,1)].mean():.4f}")
print(f"Corr. media Grupo B: {corr.loc['x4_B','x5_B']:.4f}")
print(f"Corr. media Grupo C: {corr.loc[['x6_C','x7_C','x8_C'],['x6_C','x7_C','x8_C']].values[np.triu_indices(3,1)].mean():.4f}")

---
## 2. Split train / test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print(f'Train: {X_train.shape[0]} obs  |  Test: {X_test.shape[0]} obs')

---
## 3. Ajuste de modelos

Usamos `Pipeline` con `StandardScaler` (patron del repo del profesor) para que la regularizacion
trate a todos los predictores en la misma escala.

In [ ]:
cv            = KFold(n_splits=5, shuffle=True, random_state=SEED)
alphas_grid   = np.logspace(-4, 2, 100)

# OLS
ols_pipe = Pipeline([('scaler', StandardScaler()),
                     ('model',  LinearRegression())])
ols_pipe.fit(X_train, y_train)

# Ridge con RidgeCV
ridge_cv = Pipeline([('scaler', StandardScaler()),
                     ('model',  RidgeCV(alphas=alphas_grid, cv=cv))])
ridge_cv.fit(X_train, y_train)
best_lambda_ridge = ridge_cv.named_steps['model'].alpha_
print(f'Ridge  -> lambda optimo = {best_lambda_ridge:.4f}')

# LASSO con LassoCV
lasso_cv = Pipeline([('scaler', StandardScaler()),
                     ('model',  LassoCV(alphas=alphas_grid, cv=cv,
                                        max_iter=10000, random_state=SEED))])
lasso_cv.fit(X_train, y_train)
best_lambda_lasso = lasso_cv.named_steps['model'].alpha_
print(f'LASSO  -> lambda optimo = {best_lambda_lasso:.4f}')

# Elastic-Net con ElasticNetCV (alpha en PDF = l1_ratio en sklearn)
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
en_cv = Pipeline([('scaler', StandardScaler()),
                  ('model',  ElasticNetCV(
                      l1_ratio=l1_ratios,
                      n_alphas=100,
                      cv=cv,
                      max_iter=10000,
                      random_state=SEED
                  ))])
en_cv.fit(X_train, y_train)
best_lambda_en = en_cv.named_steps['model'].alpha_
best_alpha_en  = en_cv.named_steps['model'].l1_ratio_
print(f'EN     -> lambda optimo = {best_lambda_en:.4f},  l1_ratio optimo = {best_alpha_en:.2f}')

---
## 4. Métricas en el conjunto de prueba

In [ ]:
def compute_metrics(model, X_te, y_te, feat_names, name):
    """RMSE, MAE, R2, Bias, #coefs != 0 y variables seleccionadas."""
    y_pred = model.predict(X_te)
    rmse   = np.sqrt(mean_squared_error(y_te, y_pred))
    mae    = mean_absolute_error(y_te, y_pred)
    r2     = r2_score(y_te, y_pred)
    bias   = np.mean(y_pred - y_te)
    coef   = model.named_steps['model'].coef_
    nz_idx = np.where(np.abs(coef) > 1e-6)[0]
    return {
        'Modelo': name,
        'RMSE':   round(rmse, 4),
        'MAE':    round(mae, 4),
        'R2':     round(r2, 4),
        'Bias':   round(bias, 4),
        '#betas_nz': len(nz_idx),
        'Vars_sel': [feat_names[i] for i in nz_idx],
        '_coef':   coef
    }

results = {}
for name, model in [('OLS', ols_pipe), ('Ridge', ridge_cv),
                     ('LASSO', lasso_cv), ('Elastic-Net', en_cv)]:
    results[name] = compute_metrics(model, X_test, y_test, feature_names, name)

# Tabla resumen
cols = ['Modelo', 'RMSE', 'MAE', 'R2', 'Bias', '#betas_nz']
df_metrics = pd.DataFrame([{k: v for k, v in r.items() if k in cols}
                             for r in results.values()]).set_index('Modelo')
print('=== Metricas en test ===')
display(df_metrics)

print('\n=== Variables seleccionadas (primeras 9) ===')
for name, r in results.items():
    sel_main = [v for v in r['Vars_sel'] if not v.endswith('_noise')]
    print(f'  {name:12s}: {sel_main}')

---
## 5. Análisis de coeficientes

In [ ]:
df_coef = pd.DataFrame({
    'Verdad':      beta_true,
    'OLS':         results['OLS']['_coef'],
    'Ridge':       results['Ridge']['_coef'],
    'LASSO':       results['LASSO']['_coef'],
    'Elastic-Net': results['Elastic-Net']['_coef'],
}, index=feature_names)

print('=== Coeficientes de las primeras 9 variables (grupos A, B, C + x9) ===')
display(df_coef.iloc[:9].round(4))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
model_list  = ['OLS', 'Ridge', 'LASSO', 'Elastic-Net']
model_colors = ['#888888', '#4c72b0', '#55a868', '#c44e52']
x_pos       = np.arange(9)
xlabels     = ['x1\n(A)', 'x2\n(A)', 'x3\n(A)', 'x4\n(B)', 'x5\n(B)',
               'x6\n(C)', 'x7\n(C)', 'x8\n(C)', 'x9\n(iso)']

for ax, mname, mcolor in zip(axes.flat, model_list, model_colors):
    cv_vals   = df_coef[mname].iloc[:9].values
    true_vals = df_coef['Verdad'].iloc[:9].values
    ax.bar(x_pos - 0.2, true_vals, 0.38, label='Verdad', color='#aaaaaa', alpha=0.7)
    ax.bar(x_pos + 0.2, cv_vals,   0.38, label=mname, color=mcolor, alpha=0.85)
    ax.axhline(0, color='black', linewidth=0.7)
    ax.set_xticks(x_pos); ax.set_xticklabels(xlabels, fontsize=9)
    ax.set_title(mname, fontsize=12, fontweight='bold')
    ax.set_ylabel('Coeficiente')
    ax.legend()
    ax.axvspan(-0.5, 2.5, alpha=0.05, color='blue')
    ax.axvspan(2.5,  4.5, alpha=0.05, color='green')
    ax.axvspan(4.5,  7.5, alpha=0.05, color='red')

fig.suptitle('Coeficientes estimados vs verdaderos (primeras 9 vars)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. Pregunta 1 — Número de condición de X⊤X

In [ ]:
scaler_temp   = StandardScaler()
Xtr_sc        = scaler_temp.fit_transform(X_train)
XtX           = Xtr_sc.T @ Xtr_sc
eigenvalues   = np.linalg.eigvalsh(XtX)
cond_number   = eigenvalues.max() / eigenvalues.min()

print(f'Eigenvalor maximo de XtX : {eigenvalues.max():.2f}')
print(f'Eigenvalor minimo de XtX : {eigenvalues.min():.6f}')
print(f'Numero de condicion k(XtX): {cond_number:.2e}')
print()
print('Interpretacion:')
print(f'  k > 1e4 indica alta colinealidad.')
nivel = "muy mal" if cond_number > 1e6 else "moderadamente mal"
print(f'  k = {cond_number:.2e}: el sistema esta {nivel} condicionado.')
print('  -> OLS invierte una matriz casi singular: coefs con varianza enorme.')

# Varianza de OLS
sigma2 = np.var(y_train - ols_pipe.predict(X_train))
try:
    cov_ols = sigma2 * np.linalg.inv(XtX)
    print(f'\nSD(beta_OLS) para x1_A : {np.sqrt(cov_ols[0,0]):.4f}')
    print(f'SD(beta_OLS) para x4_B : {np.sqrt(cov_ols[3,3]):.4f}')
    print(f'SD(beta_OLS) para x9   : {np.sqrt(cov_ols[8,8]):.4f}')
except np.linalg.LinAlgError:
    print('XtX es singular, OLS no tiene solucion unica.')

---
## 7. Pregunta 3 — Estabilidad de LASSO vs Elastic-Net (bootstrap)

In [ ]:
B   = 100
rng = np.random.default_rng(SEED)

lasso_sel = np.zeros((B, 3))
en_sel    = np.zeros((B, 3))

for b in range(B):
    idx = rng.choice(len(X_train), len(X_train), replace=True)
    Xb, yb = X_train[idx], y_train[idx]

    pipe_l = Pipeline([('sc', StandardScaler()),
                       ('m',  Lasso(alpha=best_lambda_lasso,
                                    max_iter=10000, random_state=SEED))])
    pipe_l.fit(Xb, yb)
    lasso_sel[b] = (np.abs(pipe_l.named_steps['m'].coef_[:3]) > 1e-6)

    pipe_e = Pipeline([('sc', StandardScaler()),
                       ('m',  ElasticNet(alpha=best_lambda_en,
                                          l1_ratio=best_alpha_en,
                                          max_iter=10000, random_state=SEED))])
    pipe_e.fit(Xb, yb)
    en_sel[b] = (np.abs(pipe_e.named_steps['m'].coef_[:3]) > 1e-6)

pi_lasso = lasso_sel.mean(axis=0)
pi_en    = en_sel.mean(axis=0)
vars_A   = ['x1_A', 'x2_A', 'x3_A']

df_stab = pd.DataFrame({'Variable': vars_A, 'pi_LASSO': pi_lasso, 'pi_EN': pi_en})
print('=== Frecuencia de seleccion en el Grupo A (B=100 bootstraps) ===')
display(df_stab.round(3))

fig, ax = plt.subplots(figsize=(6, 4))
xp = np.arange(3)
ax.bar(xp - 0.2, pi_lasso, 0.38, label='LASSO', color='#4c72b0')
ax.bar(xp + 0.2, pi_en,    0.38, label='Elastic-Net', color='#c44e52')
ax.set_xticks(xp); ax.set_xticklabels(vars_A)
ax.set_ylim(0, 1.1); ax.set_ylabel('Frecuencia de seleccion (pi_j)')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='50%')
ax.set_title('Estabilidad de seleccion en el Grupo A (B=100)'); ax.legend()
plt.tight_layout(); plt.show()

print('\nLASSO elige arbitrariamente una variable del grupo segun el remuestreo.')
print('Elastic-Net distribuye la seleccion gracias al componente L2.')

---
## 8. Pregunta 4 — Grouping Bound de Zou & Hastie

In [ ]:
sc2    = StandardScaler()
Xsc2   = sc2.fit_transform(X_train)
coef_e = en_cv.named_steps['model'].coef_
norm_y = np.linalg.norm(y_train)

pairs = [('x1_A', 'x2_A', 0, 1),
         ('x1_A', 'x3_A', 0, 2),
         ('x2_A', 'x3_A', 1, 2)]

print('=== Grouping bound de Zou & Hastie (2005) ===')
print(f'  lambda EN = {best_lambda_en:.4f}  |  ||y|| = {norm_y:.2f}')
print(f'  Cota: (1/lambda)*sqrt(2*(1-rho))*||y||')
print()
print(f'{"Par":<20} {"rho":>8} {"Cota sup.":>14} {"|bi-bj| (EN)":>16} {"OK?":>5}')
print('-' * 68)

for v1, v2, i, j in pairs:
    rho   = np.corrcoef(Xsc2[:, i], Xsc2[:, j])[0, 1]
    cota  = (1 / best_lambda_en) * np.sqrt(2 * (1 - rho)) * norm_y
    diff  = abs(coef_e[i] - coef_e[j])
    check = 'SI' if diff <= cota else 'NO'
    print(f'{v1+"-"+v2:<20} {rho:>8.4f} {cota:>14.4f} {diff:>16.4f} {check:>5}')

print()
print('Cuando rho -> 1, la cota -> 0, i.e. los coeficientes EN deben ser casi iguales.')

---
## 9. Pregunta 5 — Grupo C (irrelevante)

In [ ]:
grupo_C = [5, 6, 7]
print('=== Coeficientes del Grupo C (x6, x7, x8) ===')
for name, r in results.items():
    vals   = r['_coef'][grupo_C]
    status = 'ELIMINADO' if all(abs(v) < 1e-6 for v in vals) else 'No eliminado'
    print(f'  {name:12s}: {np.round(vals, 4)}  -> {status}')

---
## 10. Pregunta 6 — Recuperación de x9 (señal aislada)

In [ ]:
print(f'Coeficiente verdadero de x9: 1.5')
print(f'{"Modelo":<15} {"beta_x9":>10} {"Error":>10}')
print('-' * 38)
for name, r in results.items():
    est = r['_coef'][8]
    print(f'{name:<15} {est:>10.4f} {est - 1.5:>10.4f}')
print()
print('x9 no tiene colinealidad: todos los metodos la recuperan bien.')

---
## 11. Pregunta 7 — Rendimiento predictivo vs estabilidad interpretativa

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
mnames  = list(results.keys())
colors4 = ['#888888', '#4c72b0', '#55a868', '#c44e52']
rmse_v  = [results[m]['RMSE'] for m in mnames]
nz_v    = [results[m]['#betas_nz'] for m in mnames]

bars = axes[0].bar(mnames, rmse_v, color=colors4, alpha=0.85)
axes[0].set_ylabel('RMSE (test)'); axes[0].set_title('Error de prediccion')
for b, v in zip(bars, rmse_v):
    axes[0].text(b.get_x() + b.get_width()/2, v + 0.003,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=10)

axes[1].bar(mnames, nz_v, color=colors4, alpha=0.85)
axes[1].axhline(3, color='red', linestyle='--', linewidth=1, label='3 vars. verdaderas')
axes[1].set_ylabel('# coeficientes != 0'); axes[1].set_title('Sparsidad del modelo')
axes[1].legend()
for i, v in enumerate(nz_v):
    axes[1].text(i, v + 0.3, str(v), ha='center', fontsize=11)

plt.suptitle('Prediccion vs interpretabilidad', fontsize=13)
plt.tight_layout(); plt.show()

print('Resumen:')
print('  Ridge puede tener buen RMSE pero modelo denso (no interpretable).')
print('  LASSO es sparse pero inestable en el Grupo A.')
print('  Elastic-Net: mejor compromiso entre sparsidad estable y buen RMSE.')

---
## 12. Rutas de regularización (regularization path)

In [ ]:
from sklearn.linear_model import lasso_path, enet_path
from matplotlib.lines import Line2D

sc3   = StandardScaler()
Xsc3  = sc3.fit_transform(X_train)

alphas_l, coefs_l, _ = lasso_path(Xsc3, y_train, n_alphas=100)
alphas_e, coefs_e, _ = enet_path(Xsc3, y_train,
                                   l1_ratio=best_alpha_en, n_alphas=100)

cg = (['#1f77b4']*3 + ['#2ca02c']*2 + ['#d62728']*3 +
      ['#ff7f0e'] + ['#cccccc']*41)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, alp, coefs, title, ba in [
    (axes[0], alphas_l, coefs_l, 'LASSO path', best_lambda_lasso),
    (axes[1], alphas_e, coefs_e,
     f'Elastic-Net path (l1_ratio={best_alpha_en})', best_lambda_en)
]:
    for j in range(min(9, coefs.shape[0])):
        ax.plot(np.log10(alp), coefs[j], color=cg[j], lw=1.5)
    ax.axvline(np.log10(ba), color='black', linestyle='--', lw=1,
               label=f'lambda CV = {ba:.4f}')
    ax.set_xlabel('log10(lambda)'); ax.set_ylabel('Coeficiente')
    ax.set_title(title); ax.legend(fontsize=9)

legend_el = [
    Line2D([0],[0], color='#1f77b4', lw=2, label='Grupo A (x1-x3)'),
    Line2D([0],[0], color='#2ca02c', lw=2, label='Grupo B (x4-x5)'),
    Line2D([0],[0], color='#d62728', lw=2, label='Grupo C (x6-x8)'),
    Line2D([0],[0], color='#ff7f0e', lw=2, label='x9 aislada'),
]
axes[0].legend(handles=legend_el, fontsize=8, loc='upper left')
plt.suptitle('Rutas de regularizacion (primeras 9 variables)', fontsize=12)
plt.tight_layout(); plt.show()

---
## 13. Tabla de respuestas a las preguntas del PDF

In [ ]:
respuestas = [
    ('P1', 'OLS bajo colinealidad',
     'k(XtX) muy alto. OLS invierte una matriz casi singular: '
     'coefs con varianza enorme, signos erroneos e inestables.'),
    ('P2', 'Ridge hace seleccion?',
     'No. Ridge shrinkea coefs hacia cero pero NUNCA los elimina (penalizacion L2). '
     'Util para estabilidad numerica, no para seleccion de variables.'),
    ('P3', 'LASSO en Grupo A',
     'Con distintos remuestros, LASSO elige arbitrariamente una variable del grupo. '
     'Elastic-Net distribuye y estabiliza la seleccion gracias al componente L2.'),
    ('P4', 'Grouping bound',
     'Verificado numericamente: |beta_i - beta_j| <= cota, con cota -> 0 cuando rho -> 1. '
     'EN asigna coefs similares a variables muy correlacionadas.'),
    ('P5', 'Grupo C (distractores)',
     'LASSO y EN eliminan el Grupo C completamente. '
     'Ridge lo shrinkea pero lo mantiene. OLS puede darle coefs grandes.'),
    ('P6', 'x9 (senal aislada)',
     'Todos los metodos recuperan bien x9 (~1.5). Sin colinealidad, la regularizacion no dificulta.'),
    ('P7', 'Predictivo vs interpretable',
     'Ridge puede ser el mas predictivo pero es denso. '
     'LASSO es sparse pero inestable en grupos colineales. '
     'Elastic-Net: mejor compromiso (sparse + estable). RMSE bajo != interpretabilidad.')
]

df_resp = pd.DataFrame(respuestas, columns=['#', 'Pregunta', 'Respuesta'])
pd.set_option('display.max_colwidth', None)
display(df_resp.set_index('#'))